# 05 — CLI avec `argparse`

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- créer une ligne de commande avec `argparse` ;
- déclarer arguments positionnels et options ;
- gérer les types, valeurs par défaut, choix restreints ;
- comprendre `--help` auto-généré.

## Prérequis

- fonctions, structure de projet, imports ;
- logging.

Pas encore vus :

- pytest, build wheel.

## Plan

1. Pourquoi `argparse` ?
2. Premier exemple
3. Arguments positionnels
4. Options nommées
5. Types et défauts
6. Choix restreints
7. Point d'entrée complet
8. Synthèse
9. Exercices

---


## 1. Pourquoi `argparse` ?

`sys.argv` contient les arguments bruts. `argparse` en fait un **parser structuré** qui :

- valide les types ;
- gère les défauts ;
- génère `--help` automatiquement ;
- affiche un message d'erreur clair en cas de mauvaise utilisation.

Il y a des alternatives (`click`, `typer`) mais `argparse` est dans la stdlib et suffit pour la majorité des besoins.

---


## 2. Premier exemple

In [ ]:
import argparse

def construire_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog='saluer',
        description='Affiche un message de bienvenue.',
    )
    parser.add_argument('nom', help='nom de la personne à saluer')
    return parser

# Simulation (on passe les args à la main dans un notebook)
parser = construire_parser()
args = parser.parse_args(['Alice'])
print(args)

---


## 3. Arguments positionnels

Les positionnels sont **obligatoires** et reconnus par leur **position**.

In [ ]:
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('source')
parser.add_argument('destination')

args = parser.parse_args(['fichier.txt', '/tmp/copie.txt'])
print(args.source, '->', args.destination)

---


## 4. Options nommées

Les options commencent par `--` (forme longue) et éventuellement `-` (forme courte).

In [ ]:
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('--verbose', '-v', action='store_true', help='mode verbeux')
parser.add_argument('--output', '-o', default='/tmp/out.txt', help='fichier de sortie')

args = parser.parse_args(['-v', '--output', '/tmp/specifique.txt'])
print(args)

### `action='store_true'`

Transforme l'option en **drapeau** booléen. Présente ⇒ `True`, absente ⇒ `False` (par défaut).

---


## 5. Types et défauts

In [ ]:
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('--n', type=int, default=10, help='nombre de répétitions')
parser.add_argument('--seuil', type=float, default=0.5, help='seuil')

args = parser.parse_args(['--n', '5', '--seuil', '0.8'])
print(args.n, type(args.n))
print(args.seuil, type(args.seuil))

---


## 6. Choix restreints

In [ ]:
import argparse

parser = argparse.ArgumentParser()
parser.add_argument('--format', choices=['csv', 'json', 'xml'], default='json')

args = parser.parse_args(['--format', 'csv'])
print(args.format)

Tenter une valeur non prévue lève une erreur claire.

In [ ]:
try:
    parser.parse_args(['--format', 'yaml'])
except SystemExit as err:
    print('argparse a appelé sys.exit :', err)

---


## 7. Point d'entrée complet

In [ ]:
import argparse

def construire_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        prog='resume',
        description='Calcule min/max/moyenne d'une liste de nombres.',
    )
    parser.add_argument('nombres', type=float, nargs='+', help='les nombres')
    parser.add_argument('--format', choices=['texte', 'json'], default='texte')
    return parser

def resume(nombres: list[float]) -> dict[str, float]:
    return {
        'min': min(nombres),
        'max': max(nombres),
        'moyenne': sum(nombres) / len(nombres),
    }

def principal(argv: list[str] | None = None) -> int:
    parser = construire_parser()
    args = parser.parse_args(argv)
    r = resume(args.nombres)
    if args.format == 'json':
        import json
        print(json.dumps(r, indent=2))
    else:
        for cle, valeur in r.items():
            print(cle, ':', valeur)
    return 0

principal(['1', '2', '3', '4', '5'])
principal(['1', '2', '3', '--format', 'json'])

### `nargs='+'`

Exige **au moins un** argument et les capture tous dans une liste. `nargs='*'` accepte zéro ou plus.

---


## 8. Synthèse

| Besoin | Appel |
|---|---|
| Positionnel | `add_argument('nom')` |
| Option drapeau | `add_argument('--verbose', action='store_true')` |
| Option typée | `add_argument('--n', type=int, default=10)` |
| Choix | `add_argument('--fmt', choices=['csv', 'json'])` |
| Plusieurs valeurs | `add_argument('vals', nargs='+')` |
| Aide auto | Gratuit via `--help` |

### Règles

1. Un `principal(argv=None)` testable (on peut lui passer une liste en test).
2. Retourner un exit code (`0` succès, `1` erreur).
3. Coupler à `logging` pour l'affichage, pas `print` pour les messages d'état.

---


## 9. Exercices

### Exercice 1 — Un positionnel *(facile)*

Écrire `construire_parser()` qui accepte un argument positionnel `fichier`. Tester sur `['data.csv']`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_CLI_argparse", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import argparse

def construire_parser() -> argparse.ArgumentParser:
    """Parser avec un seul positionnel `fichier`."""
    p = argparse.ArgumentParser()
    p.add_argument('fichier', help='fichier à traiter')
    return p

print(construire_parser().parse_args(['data.csv']))
```

</details>

### Exercice 2 — Drapeau verbose *(facile)*

Ajouter au parser précédent une option `--verbose`/`-v` qui bascule un booléen.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_CLI_argparse", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import argparse

def construire_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser()
    p.add_argument('fichier')
    p.add_argument('--verbose', '-v', action='store_true')
    return p

print(construire_parser().parse_args(['data.csv', '-v']))
```

</details>

### Exercice 3 — Nombre typé *(facile)*

Ajouter une option `--n` de type `int`, défaut `10`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_CLI_argparse", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import argparse

def construire_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser()
    p.add_argument('fichier')
    p.add_argument('--n', type=int, default=10)
    return p

print(construire_parser().parse_args(['data.csv', '--n', '42']))
```

</details>

### Exercice 4 — Format *(moyen)*

Ajouter une option `--format` limitée à `csv`, `json`. Défaut `csv`. Écrire `principal(argv)` qui affiche le format choisi.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_CLI_argparse", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import argparse

def construire_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser()
    p.add_argument('fichier')
    p.add_argument('--format', choices=['csv', 'json'], default='csv')
    return p

def principal(argv: list[str] | None = None) -> int:
    args = construire_parser().parse_args(argv)
    print('fichier :', args.fichier)
    print('format  :', args.format)
    return 0

principal(['data.csv', '--format', 'json'])
```

</details>

### Exercice 5 — Multi-valeurs *(moyen)*

Écrire une CLI `somme` qui prend un nombre variable de floats en positionnels (`nargs='+'`) et affiche leur somme.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_CLI_argparse", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
import argparse

def principal(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser(prog='somme')
    p.add_argument('valeurs', type=float, nargs='+')
    args = p.parse_args(argv)
    print('somme :', sum(args.valeurs))
    return 0

principal(['1.5', '2.5', '3.0'])
```

</details>

### Exercice 6 — Pipeline CSV → JSON *(difficile)*

Écrire `principal(argv)` qui prend un fichier CSV en positionnel, une option `--output` (défaut `out.json`), et convertit le CSV en JSON dans le fichier de sortie (utiliser `csv.DictReader` + `json.dump`).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_CLI_argparse", exercice=6)


<details>
<summary>📖 Voir la correction</summary>

```python
import argparse
import csv
import json

def principal(argv: list[str] | None = None) -> int:
    p = argparse.ArgumentParser()
    p.add_argument('source')
    p.add_argument('--output', '-o', default='out.json')
    args = p.parse_args(argv)

    with open(args.source, 'r', encoding='utf-8', newline='') as f_in:
        donnees = list(csv.DictReader(f_in))
    with open(args.output, 'w', encoding='utf-8') as f_out:
        json.dump(donnees, f_out, ensure_ascii=False, indent=2)
    return 0

from pathlib import Path
Path('/tmp/in.csv').write_text('nom,age\nAlice,30\n', encoding='utf-8')
principal(['/tmp/in.csv', '-o', '/tmp/out.json'])
print(open('/tmp/out.json', encoding='utf-8').read())
```

</details>

---


## Ressources externes

- [`argparse`](https://docs.python.org/3/library/argparse.html)
- [Tutoriel argparse](https://docs.python.org/3/howto/argparse.html)
- [`click`](https://click.palletsprojects.com/)
- [`typer`](https://typer.tiangolo.com/)

---

## Mini-exemples supplémentaires

### Help automatique

In [ ]:
import argparse
parser = argparse.ArgumentParser(prog='demo', description='Démonstration CLI.')
parser.add_argument('fichier', help='fichier à traiter')
parser.add_argument('--verbose', '-v', action='store_true', help='verbeux')
parser.print_help()

### Action `count` — niveau de verbosité

In [ ]:
import argparse
parser = argparse.ArgumentParser()
parser.add_argument('-v', action='count', default=0)

args = parser.parse_args(['-vvv'])
print('niveau verbosité :', args.v)

### Version auto

In [ ]:
import argparse
parser = argparse.ArgumentParser()
parser.add_argument('--version', action='version', version='mon-outil 1.2.3')
try:
    parser.parse_args(['--version'])
except SystemExit:
    print('--version a bien fini le programme')

### Sous-commandes

In [ ]:
import argparse
parser = argparse.ArgumentParser(prog='git-lite')
sub = parser.add_subparsers(dest='commande', required=True)

add = sub.add_parser('add', help='ajouter un fichier')
add.add_argument('fichier')

commit = sub.add_parser('commit', help='créer un commit')
commit.add_argument('-m', '--message', required=True)

print(parser.parse_args(['add', 'README.md']))
print(parser.parse_args(['commit', '-m', 'initial']))

Chaque sous-commande a son propre parser, ses propres options. C'est le pattern de `git`, `docker`, `uv`, etc.

### Parser + `dataclass`

In [ ]:
# from dataclasses import dataclass
# 
# @dataclass
# class Args:
#     fichier: str
#     verbose: bool
# 
# def parse() -> Args:
#     p = argparse.ArgumentParser()
#     p.add_argument('fichier')
#     p.add_argument('--verbose', action='store_true')
#     raw = p.parse_args()
#     return Args(fichier=raw.fichier, verbose=raw.verbose)
print('pattern dataclass + argparse')

### `argparse` vs `click`/`typer`

- `argparse` : stdlib, aucune dépendance. Suffit pour la majorité des besoins.
- `click` : plus lisible, décorateurs, groupes.
- `typer` : `click` + annotations de type, très concis.

**Pour l'initiation, on reste sur `argparse`**.

---

## Quiz flash — vérifiez vos acquis

Ce quiz est là pour que vous vérifiiez rapidement votre compréhension avant de passer au notebook suivant. Les réponses sont dans le bloc `<details>` en dessous.


**Question 1.** Quelle méthode crée un argument ?

<details>
<summary>📖 Réponse</summary>

`parser.add_argument(...)`.

</details>

**Question 2.** Comment déclarer un drapeau booléen ?

<details>
<summary>📖 Réponse</summary>

`add_argument('--verbose', action='store_true')`.

</details>

**Question 3.** Comment exiger au moins un positionnel parmi plusieurs ?

<details>
<summary>📖 Réponse</summary>

Avec `nargs='+'`.

</details>

**Question 4.** Comment restreindre une option à un ensemble de valeurs ?

<details>
<summary>📖 Réponse</summary>

Avec `choices=[...]`.

</details>

**Question 5.** Qui génère automatiquement `--help` ?

<details>
<summary>📖 Réponse</summary>

`argparse` lui-même, à partir des `help=...` passés à `add_argument`.

</details>

---

## Cheat sheet — `argparse` patterns

In [ ]:
# Pattern minimal
import argparse

def build() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(prog='outil', description='...')
    p.add_argument('fichier', help='fichier source')
    p.add_argument('-o', '--output', default='out.txt', help='fichier cible')
    p.add_argument('-v', '--verbose', action='store_true')
    p.add_argument('--n', type=int, default=10)
    p.add_argument('--format', choices=['csv', 'json'], default='csv')
    return p

build().parse_args(['entree.txt', '-o', '/tmp/sortie.json', '--format', 'json'])

### Les 6 actions essentielles

| Action | Effet |
|---|---|
| `store` (défaut) | Range la valeur passée |
| `store_true` | Drapeau booléen (défaut `False`) |
| `store_false` | Drapeau inversé |
| `store_const` | Stocke une constante prédéfinie |
| `count` | Compte le nombre d'apparitions (`-vvv` → 3) |
| `append` | Collectionne dans une liste |

### Point d'entrée standard

In [ ]:
import argparse
import sys

def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument('valeurs', type=int, nargs='+')
    args = parser.parse_args(argv)
    print('somme :', sum(args.valeurs))
    return 0

if __name__ == '__main__':
    sys.exit(main())